In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import LabelEncoder
import io


In [12]:
from google.colab import files
uploaded = files.upload()

Saving UNSW-NB15_1_partial_binarised.csv to UNSW-NB15_1_partial_binarised (2).csv


In [3]:
df = pd.read_csv("/content/UNSW-NB15_1_partial_binarised_injected .csv")

print(df.head())

   sport dport proto state       dur  sbytes  spkts  label
0   1390    53   udp   CON  0.001055     132      2      0
1  33661  1024   udp   CON  0.036133     528      4      0
2   1464    53   udp   CON  0.001119     146      2      0
3   3593    53   udp   CON  0.001209     132      2      0
4  49664    53   udp   CON  0.001169     146      2      0


<ipython-input-3-9abaffc591d6>:1: DtypeWarning: Columns (0,1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/content/UNSW-NB15_1_partial_binarised_injected .csv")


In [4]:
evasion_data = pd.read_csv("/content/whitebox_evasion_records.csv")

print(evasion_data.head())

   sport  dport proto state       dur  sbytes  spkts
0  10880   5060   udp   INT  0.000019    1388      2
1  54585     80   tcp   FIN  0.925452    2080     16
2  11233     25   tcp   FIN  1.547788   65540     66
3  42254    445   tcp   FIN  1.512013   10976      8
4  49346     53   tcp   CON  0.001111     132      2


In [5]:
def safe_hex_to_int(x):
    try:
        return int(x, 16)
    except (ValueError, TypeError):
        return 0  # or np.nan, depending on what you want

df['sport'] = df['sport'].apply(safe_hex_to_int)
df['dport'] = df['dport'].apply(safe_hex_to_int)

In [6]:
le_proto = LabelEncoder()
le_state = LabelEncoder()

# Fit the encoders on the respective columns from df1 and df2
le_proto.fit(pd.concat([df['proto'], evasion_data['proto']]))  # Fit on combined 'proto' column
le_state.fit(pd.concat([df['state'], evasion_data['state']]))

LabelEncoder()

In [7]:
# Transform the 'proto' and 'state' columns in both dataframes using the separate encoders
df['proto'] = le_proto.transform(df['proto'])
df['state'] = le_state.transform(df['state'])

evasion_data['proto'] = le_proto.transform(evasion_data['proto'])
evasion_data['state'] = le_state.transform(evasion_data['state'])

In [8]:
print("DataFrame 1:\n", df)
print("\nDataFrame 2:\n", evasion_data)

DataFrame 1:
          sport  dport  proto  state       dur  sbytes  spkts  label
0         5008     83    120      2  0.001055     132      2      0
1       210529   4132    120      2  0.036133     528      4      0
2         5220     83    120      2  0.001119     146      2      0
3        13715     83    120      2  0.001209     132      2      0
4       300644     83    120      2  0.001169     146      2      0
...        ...    ...    ...    ...       ...     ...    ...    ...
701116       0      0    114      5  1.547788   65540     66      0
701117       0      0    114      5  1.547788   65540     66      0
701118       0      0    114      5  1.547788   65540     66      0
701119       0      0    114      5  1.547788   65540     66      0
701120       0      0    114      5  1.547788   65540     66      0

[701121 rows x 8 columns]

DataFrame 2:
    sport  dport  proto  state       dur  sbytes  spkts
0  10880   5060    120      6  0.000019    1388      2
1  54585     80   

In [9]:
# Features
X = df.drop(columns=['label'])

# Target variable
y = df['label']


In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [11]:
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

In [12]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

# Fit the scaler to the training data and transform both training and test sets
X_train_res = scaler.fit_transform(X_train_res)
X_test = scaler.transform(X_test)

print(X_train_res[:5])


[[ 0.19143628 -0.02159089  0.02478724  0.11532099  0.00632896 -0.04424244
  -0.11915693]
 [ 0.74001671  0.42717671  0.02478724  0.11532099 -0.04775777 -0.03421582
   0.18093422]
 [ 0.38243333  0.18736357  0.02478724  0.11532099 -0.04799877 -0.03624841
   0.13592055]
 [-0.76517254 -0.02176528  0.02478724  0.11532099 -0.04855713 -0.04808453
  -0.13416149]
 [-0.76517254 -0.02176528  0.02478724  0.11532099  0.0086597  -0.04424244
  -0.11915693]]


In [13]:
from sklearn.linear_model import SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier

In [14]:
classifiers = {
    "SGDClassifier": SGDClassifier(random_state=42),
    "SVM": LinearSVC(random_state=42),
    "KNN": KNeighborsClassifier(),
    "GaussianNB": GaussianNB(),
    "DecisionTree": DecisionTreeClassifier(random_state=42),
    "RandomForest": RandomForestClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42)
}

In [15]:
results = {}

for name, clf in classifiers.items():
    print(f"\n--- {name} ---")
    clf.fit(X_train_res, y_train_res)
    y_pred = clf.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')  # or 'macro'/'micro'

    print("Classification Report:")
    print(classification_report(y_test, y_pred))

    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    results[name] = {
        "accuracy": acc,
        "f1_score": f1,
    }


--- SGDClassifier ---
Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.92      0.96    135681
           1       0.27      0.89      0.41      4544

    accuracy                           0.92    140225
   macro avg       0.63      0.90      0.68    140225
weighted avg       0.97      0.92      0.94    140225

Confusion Matrix:
[[124625  11056]
 [   504   4040]]

--- SVM ---
Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.91      0.95    135681
           1       0.25      0.90      0.40      4544

    accuracy                           0.91    140225
   macro avg       0.63      0.91      0.67    140225
weighted avg       0.97      0.91      0.93    140225

Confusion Matrix:
[[123646  12035]
 [   437   4107]]

--- KNN ---
Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.99      0.99    135681
           1 

In [16]:
print("\n--- Model Comparison ---")
for model, metrics in results.items():
    print(f"{model} -> Accuracy: {metrics['accuracy']:.4f}, F1 Score: {metrics['f1_score']:.4f}")


--- Model Comparison ---
SGDClassifier -> Accuracy: 0.9176, F1 Score: 0.9380
SVM -> Accuracy: 0.9111, F1 Score: 0.9340
KNN -> Accuracy: 0.9898, F1 Score: 0.9903
GaussianNB -> Accuracy: 0.3679, F1 Score: 0.5022
DecisionTree -> Accuracy: 0.9957, F1 Score: 0.9957
RandomForest -> Accuracy: 0.9957, F1 Score: 0.9958
AdaBoost -> Accuracy: 0.9411, F1 Score: 0.9541


In [17]:
evasion_scaled = scaler.transform(evasion_data)

In [18]:
print("\n--- Whitebox Evasion Records Prediction ---")
for name, clf in classifiers.items():
    predictions = clf.predict(evasion_scaled)
    print(f"\n{name} predictions:")
    print(predictions)


--- Whitebox Evasion Records Prediction ---

SGDClassifier predictions:
[1 0 0 1 0 1 1]

SVM predictions:
[1 0 0 1 0 1 1]

KNN predictions:
[1 0 1 1 0 1 1]

GaussianNB predictions:
[1 1 1 1 0 1 1]

DecisionTree predictions:
[1 1 1 1 0 1 1]

RandomForest predictions:
[1 1 1 1 0 1 1]

AdaBoost predictions:
[1 1 1 1 0 1 1]
